# $\color{cyan}{\text{Imports and Setup}}$

## $\color{yellow}{\text{Imports}}$

In [7]:
# Import required packages
import pandas as pd
import numpy as np
import pickle
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from scipy.stats import linregress

## $\color{yellow}{\text{Setup}}$

In [8]:
with open('dashboard_catalog.pkl', 'rb') as f:
    master_data_catalog = pickle.load(f)

print("Data Loaded")

Data Loaded


In [ ]:
def create_interactive_dashboard(data_catalog):
    '''Creates an interactive explorer for comparing data sources and features.

    Args:
        data_catalog (dict): Nested mapping of analysis modes to named dataframes.
    '''

    # State dictionary tracks the last known values to prevent overwriting active filters
    state = {
        'updating': False,
        'last_mode': None,
        'last_ch': None,
        'last_x_src': None,
        'last_x_col': None,
        'last_y_src': None,
        'last_y_col': None,
        'excluded_chips': set()
    }

    all_sources = list(data_catalog['Absolute'].keys())

    # Initialises widget controls for the dashboard
    mode_toggle = widgets.ToggleButtons(options=['Absolute', 'Stage Delta', 'Intra-stage Kinetics'], style={'description_width': 'initial'})
    channel_dropdown = widgets.Dropdown(options=['Channel 1', 'Channel 2'], value='Channel 1', description='Channel:')

    x_source_drop = widgets.Dropdown(options=all_sources, value=all_sources[0], description='X Source:')
    y_source_drop = widgets.Dropdown(options=all_sources, value=all_sources[0], description='Y Source:')

    x_col_drop = widgets.Dropdown(description='X Metric:')
    y_col_drop = widgets.Dropdown(description='Y Metric:')

    # Range filters for Greater than / Less than selection based on absolute limits
    x_min_input = widgets.BoundedFloatText(description='Min X:')
    x_max_input = widgets.BoundedFloatText(description='Max X:')
    y_min_input = widgets.BoundedFloatText(description='Min Y:')
    y_max_input = widgets.BoundedFloatText(description='Max Y:')

    # Reset Buttons
    x_reset_btn = widgets.Button(description='Reset X', button_style='info', tooltip='Reset X filters to original bounds')
    y_reset_btn = widgets.Button(description='Reset Y', button_style='info', tooltip='Reset Y filters to original bounds')

    def reset_x_filters(b):

        # Restores the values to the absolute min/max bounds established during the update step
        x_min_input.value = x_min_input.min
        x_max_input.value = x_max_input.max

    def reset_y_filters(b):

        # Restores the values to the absolute min/max bounds established during the update step
        y_min_input.value = y_min_input.min
        y_max_input.value = y_max_input.max

    x_reset_btn.on_click(reset_x_filters)
    y_reset_btn.on_click(reset_y_filters)

    # Group filters into containers initially hidden (now including reset buttons)
    x_filters = widgets.HBox([x_min_input, x_max_input, x_reset_btn], layout=widgets.Layout(display='none'))
    y_filters = widgets.HBox([y_min_input, y_max_input, y_reset_btn], layout=widgets.Layout(display='none'))

    all_chips = set()

    for mode_dict in data_catalog.values():
        for src_dict in mode_dict.values():
            for df in src_dict.values():

                if not df.empty:
                    all_chips.update(df.index.dropna().astype(str).tolist())
                    
    sorted_chips = sorted(list(all_chips))

    chip_search = widgets.Text(description='Find Chip:', placeholder='Type to filter...',tooltip='Filters the list of chips below')
    
    chip_select = widgets.Select(options=sorted_chips, rows=5, description='Select Chip:')

    def filter_chips(change):

        search_term = change.new.lower()

        if not search_term:
            chip_select.options = sorted_chips
        else:
            chip_select.options = [c for c in sorted_chips if search_term in c.lower()]

    chip_search.observe(filter_chips, names='value')

    add_chip_btn = widgets.Button(description='Exclude Selected', button_style='warning', icon='minus')
    reset_chips_btn = widgets.Button(description='Reset Exclusions', button_style='info', icon='refresh')
    
    excluded_display = widgets.HTML(value="<b>Excluded Chips:</b> <i>None</i>")
    display_box = widgets.VBox([excluded_display], layout=widgets.Layout(max_height='60px', overflow='auto', margin='0px 0px 10px 0px'))

    def update_exclusion_display():

        if not state['excluded_chips']:
            excluded_display.value = "<b>Excluded Chips:</b> <i>None</i>"
        else:
            excluded_display.value = f"<b>Excluded Chips:</b> {', '.join(sorted(state['excluded_chips']))}"

    def add_exclusion(b):

        chip = chip_select.value

        if chip and chip not in state['excluded_chips']:

            state['excluded_chips'].add(chip)
            update_exclusion_display()
            plot_data()

    def reset_exclusions(b):

        state['excluded_chips'].clear()
        chip_search.value = ''
        update_exclusion_display()
        plot_data()

    # Bind the functions to the buttons
    add_chip_btn.on_click(add_exclusion)
    reset_chips_btn.on_click(reset_exclusions)

    # Group them together into a clean grid layout
    search_area = widgets.VBox([chip_search, chip_select])
    button_area = widgets.VBox([add_chip_btn, reset_chips_btn], layout=widgets.Layout(padding='0px 0px 0px 15px', justify_content='center'))
    
    exclusion_controls = widgets.VBox([display_box, widgets.HBox([search_area, button_area])], layout=widgets.Layout(border='1px solid #ddd', padding='10px', margin='10px 0px'))

    out = widgets.Output()

    def update_sources(*args):
        '''Updates available data sources depending on the selected channel (removes Standard Curve for Ch2).'''

        if state['updating']:
            return

        state['updating'] = True

        ch = channel_dropdown.value

        # Standard Curve missing from Channel 2
        valid_sources = [s for s in all_sources if s != 'Standard Curve'] if ch == 'Channel 2' else all_sources

        curr_x, curr_y = x_source_drop.value, y_source_drop.value

        x_source_drop.options = valid_sources
        y_source_drop.options = valid_sources

        x_source_drop.value = curr_x if curr_x in valid_sources else valid_sources[0]
        y_source_drop.value = curr_y if curr_y in valid_sources else valid_sources[0]

        state['updating'] = False
        update_dropdowns()

    def update_dropdowns(*args):
        '''Refreshes feature dropdown options after the selected sources change.'''

        if state['updating']:
            return

        state['updating'] = True

        mode = mode_toggle.value
        ch = channel_dropdown.value
        x_src, y_src = x_source_drop.value, y_source_drop.value

        if x_src and y_src:

            # Extracts columns directly from the nested dictionary structure
            x_cols = list(data_catalog[mode][x_src][ch].columns)
            y_cols = list(data_catalog[mode][y_src][ch].columns)

            # Extracts columns selected
            curr_x_col = x_col_drop.value
            curr_y_col = y_col_drop.value

            # Updates the X metric dropdown options
            x_col_drop.options = x_cols

            # Updates the Y metric dropdown options
            y_col_drop.options = y_cols

            # Resets the X metric value if the current selection is invalid
            x_col_drop.value = curr_x_col if curr_x_col in x_cols else (x_cols[0] if x_cols else None)

            # Resets the Y metric value if the current selection is invalid
            y_col_drop.value = curr_y_col if curr_y_col in y_cols else (y_cols[0] if y_cols else None)

        state['updating'] = False
        update_filter_bounds()

    def update_filter_bounds(*args):
        '''Dynamically sets the absolute min/max limits ONLY for the axes that were just modified.'''

        if state['updating']:
            return

        state['updating'] = True

        mode, ch = mode_toggle.value, channel_dropdown.value
        x_src, y_src = x_source_drop.value, y_source_drop.value
        x_col, y_col = x_col_drop.value, y_col_drop.value

        # Determine exactly which axis datasets changed to avoid wiping custom filters on the other axis
        needs_x_update = (mode != state.get('last_mode') or ch != state.get('last_ch') or x_src != state.get('last_x_src') or x_col != state.get('last_x_col'))
        needs_y_update = (mode != state.get('last_mode') or ch != state.get('last_ch') or y_src != state.get('last_y_src') or y_col != state.get('last_y_col'))

        if needs_x_update:

            x_filters.layout.display = 'flex'

            if x_col:

                df_x = data_catalog[mode][x_src][ch]

                if not df_x.empty and x_col in df_x.columns:

                    x_data = df_x[x_col].dropna()

                    if not x_data.empty:

                        x_min, x_max = float(x_data.min()), float(x_data.max())

                        if x_min == x_max:
                            x_max += 1e-9

                        # Create a safe large finite number
                        LARGE_NUM = 1e30 

                        x_min_input.min, x_max_input.max = -LARGE_NUM, LARGE_NUM
                        x_max_input.min, x_min_input.max = -LARGE_NUM, LARGE_NUM

                        x_min_input.value, x_max_input.value = x_min, x_max
                        x_min_input.min, x_min_input.max = x_min, x_max
                        x_max_input.min, x_max_input.max = x_min, x_max

            # Save the new X state
            state['last_x_src'], state['last_x_col'] = x_src, x_col

        if needs_y_update:

            y_filters.layout.display = 'flex'

            if y_col:

                df_y = data_catalog[mode][y_src][ch]

                if not df_y.empty and y_col in df_y.columns:

                    y_data = df_y[y_col].dropna()

                    if not y_data.empty:

                        y_min, y_max = float(y_data.min()), float(y_data.max())

                        if y_min == y_max:
                            y_max += 1e-9

                        # Create a safe large finite number
                        LARGE_NUM = 1e30

                        y_min_input.min, y_max_input.max = -LARGE_NUM, LARGE_NUM
                        y_max_input.min, y_min_input.max = -LARGE_NUM, LARGE_NUM

                        y_min_input.value, y_max_input.value = y_min, y_max
                        y_min_input.min, y_min_input.max = y_min, y_max
                        y_max_input.min, y_max_input.max = y_min, y_max

            # Save the new Y state
            state['last_y_src'], state['last_y_col'] = y_src, y_col

        # Save overarching state
        state['last_mode'], state['last_ch'] = mode, ch

        state['updating'] = False
        plot_data()

    def get_joined_data(mode, x_src, y_src, x_col, y_col, channel_label):
        '''Joins selected dashboard fields and applies an optional channel filter.

        Args:
            mode (str): Analysis mode selected in the dashboard.
            x_src (str): Name of the requested X data source.
            y_src (str): Name of the requested Y data source.
            x_col (str): Feature column selected for the X-axis.
            y_col (str): Feature column selected for the Y-axis.
            channel_label (str): Optional channel label used to filter rows.

        Returns:
            pd.DataFrame: Joined, complete observations for the selected fields.
        '''

        # Retrieves the specific dataframes from the catalog using the channel label

        df_x = data_catalog[mode][x_src][channel_label].copy()
        df_y = data_catalog[mode][y_src][channel_label].copy()

        # Checks whether either dataframe is empty
        if df_x.empty or df_y.empty:
            return pd.DataFrame()

        # Ensures all columns are strings for safe merging
        df_x.columns = df_x.columns.astype(str)
        df_y.columns = df_y.columns.astype(str)
        x_col_str, y_col_str = str(x_col), str(y_col)

        # Inner joins the dataframes on their index
        df_merged = pd.merge(df_x[[x_col_str]], df_y[[y_col_str]], left_index=True, right_index=True, how='inner', suffixes=('_x', '_y'))

        # Handles column renaming if X and Y columns share the same name
        actual_x_col = x_col_str + '_x' if x_col_str == y_col_str else x_col_str
        actual_y_col = y_col_str + '_y' if x_col_str == y_col_str else y_col_str
        df_merged = df_merged.rename(columns={actual_x_col: 'x_val', actual_y_col: 'y_val'})

        return df_merged.replace([np.inf, -np.inf], np.nan).dropna()

    def plot_data(*args):
        '''Creates the dashboard scatter plot and conditionally filters Standard Curve data.'''

        # Prevents redundant plotting cycles during chained updates
        if state['updating']:
            return

        with out:

            # Clears the previous output before rendering the new plot
            out.clear_output(wait=True)

            mode, channel = mode_toggle.value, channel_dropdown.value
            x_src, y_src = x_source_drop.value, y_source_drop.value
            x_col, y_col = x_col_drop.value, y_col_drop.value

            # Displays an error if the selected columns are invalid
            if not x_col or not y_col:
                print('Please select valid metrics to plot.')

                return

            # Creates a new Plotly figure for the visualisation
            fig = go.Figure()

            # Sets the active channels based on the dropdown selection
            channels_to_plot = ['Channel 1', 'Channel 2'] if channel == 'Both' else [channel]
            colors = {'Channel 1': '#1f77b4', 'Channel 2': '#ff1e0e'}
            plotted_any = False

            # Loops through each channel to plot its data
            for ch in channels_to_plot:

                # Safely attempts to join and extract the necessary data
                try:
                    plot_df = get_joined_data(mode, x_src, y_src, x_col, y_col, ch)
                except Exception as e:
                    continue

                # Create a baseline mask allowing all data through
                mask = pd.Series(True, index=plot_df.index)

                # Apply X and Y filters universally
                mask &= (plot_df['x_val'] >= x_min_input.value) & (plot_df['x_val'] <= x_max_input.value)
                mask &= (plot_df['y_val'] >= y_min_input.value) & (plot_df['y_val'] <= y_max_input.value)

                # Apply the Chip Exclusion filter
                if state['excluded_chips']:
                    mask &= ~plot_df.index.astype(str).isin(state['excluded_chips'])

                plot_df = plot_df[mask]

                # Skips the channel if there are insufficient points for plotting
                if len(plot_df) < 2:
                    continue

                plotted_any = True
                x_data, y_data = plot_df['x_val'], plot_df['y_val']

                # Adds a scatter trace for the data points
                fig.add_trace(go.Scatter(x=x_data, y=y_data, mode='markers', name=f'{ch}', marker=dict(size=8, opacity=0.7, color=colors[ch], line=dict(width=1, color='DarkSlateGrey')), text=plot_df.index, hovertemplate='Chip ID: %{text}<br>X: %{x:.4f}<br>Y: %{y:.4f}<extra></extra>'))

                # Calculates and plots a linear regression fit if variance exists
                if x_data.nunique() > 1:

                    slope, intercept, r_value, p_value, std_err = linregress(x_data, y_data)
                    x_fit = np.linspace(x_data.min(), x_data.max(), 100)
                    y_fit = slope * x_fit + intercept

                    fig.add_trace(go.Scatter(x=x_fit, y=y_fit, mode='lines', name=f'{ch} Fit (r={r_value:.3f}, R^2={r_value**2:.3f})', line=dict(color=colors[ch], dash='dash', width=2), hoverinfo='skip'))

            # Reports an error if no valid data was found for any channel
            if not plotted_any:
                print('Not enough matching chip records to plot these selections within the specified filter bounds.')

                return

            # Formats and displays the final interactive dashboard
            title = f'{y_src} [{y_col}] vs {x_src} [{x_col}]'
            fig.update_layout(title=title, xaxis_title=f'{x_src} : {x_col}', yaxis_title=f'{y_src} : {y_col}', template='plotly_white', height=600, margin=dict(l=40, r=40, t=60, b=40), hovermode='closest')
            fig.show()

    # Binds appropriate observers to orchestrate the rendering cascade
    channel_dropdown.observe(update_sources, 'value')
    mode_toggle.observe(update_dropdowns, 'value')
    x_source_drop.observe(update_dropdowns, 'value')
    y_source_drop.observe(update_dropdowns, 'value')
    x_col_drop.observe(update_filter_bounds, 'value')
    y_col_drop.observe(update_filter_bounds, 'value')

    # Bind the filters directly to the plot_data step, bypassing data re-extraction
    x_min_input.observe(plot_data, 'value')
    x_max_input.observe(plot_data, 'value')
    y_min_input.observe(plot_data, 'value')
    y_max_input.observe(plot_data, 'value')

    # Structure UI components
    controls = widgets.VBox([mode_toggle, channel_dropdown, exclusion_controls, widgets.HBox([x_source_drop, x_col_drop]), x_filters, widgets.HBox([y_source_drop, y_col_drop]), y_filters])

    # Displays the dashboard elements
    display(widgets.HTML(f"<h3 style='margin-bottom:0px; color:#0000FF;'>Master Chip Comparison Dashboard</h3>"))
    display(controls, out)

    # Initialises the widget
    update_sources()

# $\color{cyan}{\text{Dashboard Display}}$

In [23]:
# Launches the interactive cross-stage data exploration dashboard
create_interactive_dashboard(master_data_catalog)

HTML(value="<h3 style='margin-bottom:0px; color:#0000FF;'>Master Chip Comparison Dashboard</h3>")

Output()